In [1]:
defe

NameError: name 'defe' is not defined

In [1]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os
# 
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geoalchemy2 import Geometry
from shapely.geometry import LineString

import json
from pathlib import Path

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [2]:
PATH_SHPS = RUTA_UNIDAD_ONE_DRIVE + r"\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA\SHP_RECORRIDOS"

In [3]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_machine_usage_completo_df(ruta_archivo):
    """
    Lee el JSON, extrae 'MachineUsage' en un DataFrame e integra
    las columnas repetidas de ClientName, FarmName y FieldName.
    """
    try:
        # 1. Leer el archivo JSON
        with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
            datos = json.load(archivo)
        
        # 2. Extraer los datos generales del encabezado
        cliente = datos.get("ClientName", "Desconocido")
        finca = datos.get("FarmName", "Desconocido")
        lote = datos.get("FieldName", "Desconocido") # FieldName suele referirse al lote/campo
        
        # 3. Extraer la sección MachineUsage
        machine_usage_data = datos.get("MachineUsage")
        
        if not machine_usage_data:
            print("Aviso: No se encontró la sección 'MachineUsage' o está vacía.")
            return pd.DataFrame()
            
        # 4. Convertir a DataFrame (orientando por filas usando las llaves "1", "2")
        df = pd.DataFrame.from_dict(machine_usage_data, orient='index')
        df = df.reset_index().rename(columns={'index': 'IdInterno'})
        
        # 5. Agregar los datos generales de forma repetida a cada fila
        df['ClientName'] = cliente
        df['FarmName'] = finca
        df['FieldName'] = lote
        
        # Opcional: Reordenar las columnas para que los datos generales queden al principio
        columnas_ordenadas = ['ClientName', 'FarmName', 'FieldName', 'IdInterno', 
                              'MachineId', 'MachineSerial', 'PrincipalId', 'OperatorId', 'OperatorName']
        df = df[columnas_ordenadas]
        
        return df

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en {ruta_archivo}")
        return None
    except json.JSONDecodeError:
        print("Error: El archivo no es un JSON válido.")
        return None

def obtener_archivos_db():
    query = text(f"SELECT nombre_archivo FROM datos_cosecha.registro_de_archivos")
    try:
        engine = obtener_engine()
        with engine.connect() as conn:
            result = conn.execute(query)
            # Retornamos un set para búsquedas O(1) en Python
            return [row[0] for row in result]
    except Exception as e:
        print(f"Error al consultar la tabla de registro: {e}")
        return []

def obtener_archivos_locales(path_local):
    try:
        # Listamos archivos que terminen en .shp y sean archivos (no carpetas)
        archivos_shp = [
            archivo for archivo in os.listdir(path_local) 
            if archivo.lower().endswith('.shp') and os.path.isfile(os.path.join(path_local, archivo))
        ]
        return archivos_shp
    except FileNotFoundError:
        print(f"Error: La ruta '{path_local}' no existe.")
        return []
    except Exception as e:
        print(f"Error al leer la carpeta: {e}")
        return []

def cargar_archivo_local(ruta_shp):
    gdf = gpd.read_file(ruta_shp)
    gdf['IsoTime'] = pd.to_datetime(gdf['IsoTime'])
    #gdf.columns = [c.lower() for c in gdf.columns]
    gdf.columns = gdf.columns.str.lower()
    gdf = gdf.rename(columns={'geometry': 'geom'})
    gdf = gdf.set_geometry('geom')
    gdf = gdf.to_crs(epsg=32720)
    columnas_db = ['distance',
                   'swathwidth',
                   'vryldrcane',
                   'sectionid',
                   'crop',
                   'trash',
                   'time',
                   'heading',
                   'variety',
                   'elevation',
                   'isotime',
                   'machine',
                   'fuel',
                   'vehiclspeed',
                   'producthash', 
                   'geom']
    gdf = gdf[columnas_db].copy()
    return gdf

def obtener_id_lote(conn, nombre_archivo):
    """Busca el ID de la tabla maestra usando el nombre del archivo."""
    try:
        query = text("""
            SELECT id FROM datos_cosecha.fechas 
            WHERE nombre_archivo = :nom_file
        """)
        res = conn.execute(query, {'nom_file': nombre_archivo}).fetchone()
        return res[0] if res else None
    except (IndexError, ValueError):
        return None

def cargar_puntos_shps(ruta_carpeta):
    """Función principal que coordina la lectura y carga a Postgres."""
    engine = obtener_engine()
    pendientes = obtener_lotes_pendientes() # Tu función que filtra puntos_cargados IS FALSE
    for nombre_archivo in pendientes:
        ruta_shp = os.path.join(ruta_carpeta, f"{nombre_archivo}.shp")
        if not os.path.exists(ruta_shp):
            print(f"⚠️ Archivo no encontrado: {nombre_archivo}")
            continue
        try:
            with engine.begin() as conn:
                # 1. Identificar el lote
                lote_id = obtener_id_lote(conn, nombre_archivo)
                if lote_id is None:
                    print(f"❌ No existe registro maestro para: {nombre_archivo}")
                    continue
                # 2. Preparar los datos
                df_final = preparar_geodataframe_utm(ruta_shp, lote_id)
                # 3. Insertar detalles
                df_final.to_postgis(
                    'recorridos_cosecha', 
                    conn, 
                    schema='datos_cosecha', 
                    if_exists='append'
                )
                # 4. Actualizar estado
                conn.execute(text("""
                    UPDATE datos_cosecha.fechas 
                    SET puntos_cargados = TRUE 
                    WHERE id = :id
                """), {'id': lote_id})
                print(f"✅ {nombre_archivo}: {len(df_final)} puntos cargados.")
        except Exception as e:
            print(f"❌ Error procesando {nombre_archivo}: {str(e)}")

def registrar_log_archivo(nombre_archivo, cantidad_filas):
    query = text("""
        INSERT INTO datos_cosecha.registro_de_archivos 
        (nombre_archivo, fecha_registro, cantidad_registros, maquina)
        VALUES (:nombre, CURRENT_DATE, :cantidad, :alias)
    """)
    try:
        engine = obtener_engine()
        with engine.begin() as conn:
            conn.execute(query, {"nombre": nombre_archivo, "cantidad": cantidad_filas, "alias": ""})
        return True
    except Exception as e:
        print(f"Error al registrar el log del archivo: {e}")
        return False

def insertar_datos_cosecha(gdf, nombre_archivo):
    if gdf.empty:
        print("El GeoDataFrame está vacío.")
        return
    # 1. Nombre de la tabla temporal
    temp_table = "temp_recorrido_cosechadora"
    try:
        engine = obtener_engine()
        with engine.begin() as conn:
            # 2. Cargar el GDF a una tabla temporal (sin las restricciones de la original)
            # Usamos if_exists='replace' para asegurar que la temporal esté limpia
            gdf.to_postgis(temp_table, conn, if_exists='replace', index=False)
            # 3. Insertar desde la temporal a la real usando ON CONFLICT DO NOTHING
            # Esto valida contra el UNIQUE CONSTRAINT (machine, isotime)
            query = text(f"""
                INSERT INTO datos_cosecha.recorrido_cosechadora 
                (distance, swathwidth, vryldrcane, sectionid, crop, trash, time, 
                 heading, variety, elevation, isotime, machine, fuel, 
                 vehiclspeed, producthash, distancia_real, diferencia_distancia, geom, alias, lote, propiedad, operador)
                SELECT 
                    distance, swathwidth, vryldrcane, sectionid, crop, trash, time, 
                    heading, variety, elevation, isotime, machine, fuel, 
                    vehiclspeed, producthash, distancia_real, diferencia_distancia, geom, alias, lote, propiedad, operador
                FROM {temp_table}
                ON CONFLICT (alias, isotime) DO NOTHING;
            """)
            result = conn.execute(query)
            # 4. Eliminar la tabla temporal
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_table}"))
        registrar_log_archivo(nombre_archivo, len(gdf))
        print(f"Inserción completada. Filas procesadas: {len(gdf)}")
        return True
    except Exception as e:
        print(f"Error durante la inserción: {e}")
        return False

In [4]:
archivos_procesados = obtener_archivos_db()
print('Lista de archivos procesados en base de datos:', len(archivos_procesados))

archivos_locales = obtener_archivos_locales(PATH_SHPS)
print('Lista de archivos en carpera local:', len(archivos_locales))

archivos_pendientes = list(set(archivos_locales) - set(archivos_procesados))
print('Lista de archivos en pendientes:', len(archivos_pendientes))

Lista de archivos procesados en base de datos: 0
Lista de archivos en carpera local: 3
Lista de archivos en pendientes: 3


In [5]:
archivos_pendientes

['AGROPECUARIA_EL PARAISO_PS_L10_Harvest_2026-06-30_01.shp',
 'AGROPECUARIA_EL PARAISO_PS_L9B_Harvest_2026-06-29_02.shp',
 'AGROPECUARIA_EL PARAISO_PS_L9A_Harvest_2026-06-28_00.shp']

In [6]:
for archivo in archivos_pendientes:
    # sacar solo nombre, sin extencion
    file_name = archivo.split(".")[0]
    # nombre de json
    file_json = os.path.join(PATH_SHPS, file_name + '-Deere-Metadata.json')
    # cargar y formatear datos de json
    df_maquinas = obtener_machine_usage_completo_df(file_json)
    # seleccion de columnas deseadas
    df_maquinas = df_maquinas[['ClientName', 'FarmName', 'FieldName', 'IdInterno', 'MachineSerial', 'OperatorName']]
    # cambio de nombre de columnas
    nuevos_nombres = {
        'ClientName': 'canero',
        'FarmName': 'propiedad',
        'FieldName': 'lote',
        'MachineSerial': 'alias',
        'OperatorName': 'operador'
    }
    df_maquinas = df_maquinas.rename(columns=nuevos_nombres)
    # ruta de archivo .shp
    ruta_shp = os.path.join(PATH_SHPS, archivo)
    # cargar archivo shp
    recorridos_shp = cargar_archivo_local(ruta_shp)
    # convertir a numero
    recorridos_shp['machine'] = pd.to_numeric(recorridos_shp['machine'])
    df_maquinas['IdInterno'] = pd.to_numeric(df_maquinas['IdInterno'])
    # merge de datos
    recorridos_shp_completo = pd.merge(
        recorridos_shp,           # El DataFrame de la izquierda (el principal)
        df_maquinas,              # El DataFrame de la derecha (los datos a agregar)
        left_on='machine',        # Columna llave en recorridos_shp
        right_on='IdInterno',     # Columna llave en df_maquinas
        how='left'                # 'left' mantiene todos los recorridos aunque falte alguna máquina
    )
    # eliminar IdInterno
    recorridos_shp_completo.drop(columns=['IdInterno'], inplace=True)

    lista_maquinas = list(set(recorridos_shp_completo['machine']))
    
    for maq in lista_maquinas:
        
        recorridos_maquina = recorridos_shp_completo[recorridos_shp_completo['machine'] == maq].copy()
        
        # 1. Preparación: Convertir a datetime y ordenar cronológicamente
        recorridos_maquina['isotime'] = pd.to_datetime(recorridos_maquina['isotime'])
        recorridos_maquina = recorridos_maquina.sort_values('isotime').reset_index(drop=True)
        # 2. Creamos una columna auxiliar con la geometría del punto ANTERIOR
        recorridos_maquina['prev_geom'] = recorridos_maquina['geom'].shift(1)
        # 3. Filtramos la primera fila (ya que no tiene un punto anterior para formar una línea)
        df_segmentos = recorridos_maquina.dropna(subset=['prev_geom']).copy()
        # 4. Generamos los segmentos (LineString) uniendo el punto anterior con el actual
        # Al hacerlo así, la fila mantiene sus datos originales (los del punto de destino)
        df_segmentos['geom'] = df_segmentos.apply(
            lambda x: LineString([x['prev_geom'], x['geom']]), axis=1
        )
        # 5. Limpieza: Eliminamos la columna auxiliar
        df_segmentos = df_segmentos.drop(columns=['prev_geom'])
    
        df_segmentos['distancia_real'] = df_segmentos.geom.length
        df_segmentos['diferencia_distancia'] = (df_segmentos['distance'] - df_segmentos['distancia_real']).abs()

        estado = insertar_datos_cosecha(df_segmentos, archivo)
        print(estado, 'datos registrados de MAQUINA: ', maq, 'ARCHIVO:', archivo)

Inserción completada. Filas procesadas: 10934
True datos registrados de MAQUINA:  1 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L10_Harvest_2026-06-30_01.shp
Inserción completada. Filas procesadas: 35106
True datos registrados de MAQUINA:  1 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L9B_Harvest_2026-06-29_02.shp
Inserción completada. Filas procesadas: 11186
True datos registrados de MAQUINA:  2 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L9B_Harvest_2026-06-29_02.shp
Inserción completada. Filas procesadas: 35808
True datos registrados de MAQUINA:  1 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L9A_Harvest_2026-06-28_00.shp
Inserción completada. Filas procesadas: 13437
True datos registrados de MAQUINA:  2 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L9A_Harvest_2026-06-28_00.shp
Inserción completada. Filas procesadas: 2581
True datos registrados de MAQUINA:  3 ARCHIVO: AGROPECUARIA_EL PARAISO_PS_L9A_Harvest_2026-06-28_00.shp
Inserción completada. Filas procesadas: 5070
True datos registrados de MAQUINA:  4 ARCHIVO: AGROPECUA